In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [60]:
# import data
banking_marketing_train_original = pd.read_csv('banking_marketing_train.csv', delimiter=';', quotechar='"')
banking_marketing_test_original = pd.read_csv('banking_marketing_test.csv', delimiter=';', quotechar='"')

In [75]:
# import data
banking_marketing_train = pd.read_csv('banking_marketing_train.csv', delimiter=';', quotechar='"')
banking_marketing_test = pd.read_csv('banking_marketing_test.csv', delimiter=';', quotechar='"')

## create a feature: cost
it is related to campaign, duration, contact

duration * campaign / 3600 * 38 - us dollar


from this source: https://www.magellan-solutions.com/blog/cost-of-telemarketing/

In [ ]:
banking_marketing_train.columns

In [ ]:
# Count the frequency of each unique value in the 'contact' column
contact_counts = banking_marketing_train['contact'].value_counts()

# Display the result
print(contact_counts)

In [ ]:
banking_marketing_train['cost'] = banking_marketing_train['duration'] * banking_marketing_train['campaign'] / 3600 * 38


In [ ]:
# Check for rows where 'contact' is 'unknown'
unknown_contact_indices = banking_marketing_train[banking_marketing_train['contact'] == 'unknown'].index

# Loop over rows with 'contact' == 'unknown'
for idx in unknown_contact_indices:
    # Ensure there's a previous row and a next row
    if idx > 0 and idx < len(banking_marketing_train) - 1:
        previous_row_cost = banking_marketing_train_encoded.loc[idx - 1, 'cost']
        next_row_cost = banking_marketing_train_encoded.loc[idx + 1, 'cost']
        
        # Calculate the average of the previous and next row's 'cost'
        average_cost = (previous_row_cost + next_row_cost) / 2
        
        # Assign the average to the current row's 'cost'
        banking_marketing_train_encoded.loc[idx, 'cost'] = average_cost

In [ ]:
# Drop missing values from 'cost' column (if any) to ensure clean plotting
clean_cost_data = banking_marketing_train['cost'].dropna()

# Plot the distribution of the 'cost' column
sns.histplot(clean_cost_data, kde=True, color='skyblue', bins=30)

# Set plot labels and title
plt.title('Distribution of Cost')
plt.xlabel('Cost')
plt.ylabel('Density')

# Show the plot
plt.show()


In [ ]:
## CLV = Customer Lifetime Value = Customer Value x Average Customer Lifespan
# Purchase Frequency 
# using previous and campaign (higher values indicate more engagement)
df['purchase_frequency'] = df['previous'] / (df['previous'] + df['campaign'])

# Customer Lifespan
# assume they start working at 25
df['customer_lifespan'] = (df['age'] - 25) * 365 
df['customer_lifespan'] += df['pdays'].apply(lambda x: 1 if x == -1 else x)

# Estimate Revenue per Purchase
# Historical success rate: How often y = yes among past engagements.
# avg_revenue_per_purchase from Xinzi 'deposit_amount'
df['success_rate'] = df.groupby('job')['y'].transform(lambda x: x.eq('yes').mean())
df['avg_revenue'] = df['success_rate'] * df['deposit_amount']

# calculate CLV
df['CLV'] = df['purchase_frequency'] * df['customer_lifespan'] * df['avg_revenue']

# display the first few rows 

In [ ]:
## Conversion Rate
# Convert 'y' to binary (1 for 'yes', 0 for 'no')
df['conversion_binary'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Calculate conversion rate (success rate per contact attempt)
df['conversion_rate'] = df['conversion_binary'] / df['campaign']

# Display the first few rows
print(df[['y', 'campaign', 'conversion_rate']].head()) 

In [ ]:
# CAC:
# Use campaign (number of contacts), previous (number of past contacts), poutcome (success or failure), y (whether they subscribed)

df['CAC'] = (df['campaign'] * 'cost') / df['y'].apply(lambda x: 1 if x == 'yes' else 0)


In [ ]:
# ROI = (CLV-CAC)/CAC
df['ROI'] = (df['CLV'] - df['CAC']) / df['CAC']


In [ ]:
# Select Features (X) and Target (y)
X = df[['CAC', 'CLV', 'conversion_rate']]
y = df['ROI']

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on Test Set
y_pred = model.predict(X_test)

# Model Evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Coefficients:", model.coef_)
print("Intercept:", model.intercept_)
print("Mean Squared Error:", mse)
print("R² Score:", r2)

# Predict ROI for a new campaign
new_campaign = np.array([[500, 5000, 0.07]])  # Example: CAC=500, CLV=5000, CR=0.07
predicted_roi = model.predict(new_campaign)
print("Predicted ROI for new campaign:", predicted_roi[0])

# MODEL

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = banking_marketing_train_encoded.copy()

# CAC（客户获取成本）
df['CAC'] = df['campaign'] * df['cost'] / df['conversion_binary'].replace(0, np.nan)

# ROI = (CLV - CAC) / CAC
df['ROI'] = (df['CLV'] - df['CAC']) / df['CAC']

# delete missing columns
df.dropna(subset=['ROI'], inplace=True)

# choose x&y
X = df[['CAC', 'CLV', 'conversion_rate']]
y = df['ROI']

### linear regression
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred_lin = lin_model.predict(X_test)

# evaluation?
mse_lin = mean_squared_error(y_test, y_pred_lin)
r2_lin = r2_score(y_test, y_pred_lin)
print("LR_MSE:", mse_lin)
print("R^2_LR:", r2_lin)

### XGBoost
xgb_model = XGBRegressor(objective="reg:squarederror", n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

# XGBoost evaluation
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)
print("XGBoost_MSE:", mse_xgb)
print("R^2_XB:", r2_xgb)

# choose the best
best_model = "XGBoost" if r2_xgb > r2_lin else "Linear Regression"
print(f"best: {best_model}")

# example: redict ROI
new_campaign = np.array([[500, 5000, 0.07]])  # CAC=500, CLV=5000, CR=0.07
predicted_roi = xgb_model.predict(new_campaign) if best_model == "XGBoost" else lin_model.predict(new_campaign)
print("ROI:", predicted_roi[0])


In [ ]:
# ROI classification：
# ROI > 1 → high ROI
# 0 < ROI ≤ 1 → fair ROI
# ROI ≤ 0 → low ROI